In [ ]:
# Look at raw data, look at derived data, examine the MOST computations ( near bottom of NB. These should align with train_offshore_models_mvco.py call with
# code in library mo.py. One could use this space to try different numerical solvers)
# 
# 
# Import all the libraries used in this notebook
# from pvlib.solarposition import get_solarposition
import datetime
import os
from glob import glob
from os.path import join

from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import plotly
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from  plotly.graph_objs import *
import plotly.io as pio
pio.renderers.default = 'iframe'

print(os.environ["CONDA_DEFAULT_ENV"])  # Check the name of the current Conda environment

In [ ]:
mvco_raw= pd.read_csv('data/eastData/qced_MVCO_ocn_sonic_vaisala_QC_all_data.2004-2023.csv')


In [ ]:
# Set pandas to not cut out the middle of df in the display results
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows (you can set it to a specific number if you want)

print("Total datapoints/timestamps (20 min intervals): {0} \n".format(len(mvco_raw)))
print("Column Name                      Percentage of Missing data\n")
print(mvco_raw.isna().sum()/len(mvco_raw)) # shows the percentage of data missing in the corresponding column/variable

In [ ]:
## MVCO QC file post-processed  

In [ ]:
# mvco= pd.read_csv('mvco_mlsl_qc.csv')
mvco = pd.read_csv("data/mvco_mlsl_qc.csv", encoding="utf-8", delimiter=",")
#mvco = pd.read_csv("/glade/derecho/scratch/dettling/mlsl.csv")
mvco["Time"] = pd.to_datetime(mvco["Time"])
mvco.index = mvco["Time"]
print(mvco.columns)
print(len(mvco.index))

In [ ]:
print(len(mvco))
print(mvco.isna().sum()/len(mvco))

In [ ]:
#
# Define this time series layout once, will use again in time series plots in rest of notebook
#
timeSeriesLayout = dict(
    title="Time Series with Rangeslider",
    xaxis=dict(
        rangeselector=dict(
            buttons=list(
                [
                    dict(count=1, label="1d", step="day", stepmode="forward"),
                    dict(count=7, label="1w", step="week", stepmode="forward"),
                    dict(count=30, label="1m", step="month", stepmode="forward"),
                    dict(step="all"),
                ]
            )
        ),
        rangeslider=dict(),
        type="date",
    ),
    width=1200,  # Custom width in pixels
    height=800,  # Custom height in pixels
)

layout = timeSeriesLayout



In [ ]:
def drawTraces(*args):
    data = [*args]
    plotly.offline.iplot(data, filename="basic-line-plot")
    layout = timeSeriesLayout

In [ ]:
trace1 = go.Scatter(
    x=mvco.index,
    y=mvco["azimuth:0_m:degrees"],
    name="azimuth",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace2 = go.Scatter(
    x=mvco.index,
    y=mvco["zenith:0_m:degrees"],
    name="zenith",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace3 = go.Scatter(
    x=mvco.index,
    y=mvco["GHI:0_m:W_m-2"],
    name="GHI",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
# These can be turned on and off by clicks on the legend
drawTraces(*[trace1,trace2,trace3])

In [ ]:
trace0 = go.Scatter(
    x=mvco.index,
    y=mvco["momentum_flux:18.4_m:m2_s-2"],
    name="mf",
    line=dict(color="red"),
    connectgaps=False,
    opacity=0.5,
)
trace0b = go.Scatter(
    x=mvco.index,
    y=mvco["MOST_momentum_flux:18.4_m:m2_s-2"],
    name="MOSTmf",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace0c = go.Scatter(
    x=mvco.index,
    y=mvco["MOST_chopped_momentum_flux:18.4_m:m2_s-2"],
    name="MOSTchoppedMf",
    line=dict(color="cyan"),
    connectgaps=False,
    opacity=0.5,
)
trace0d = go.Scatter(
    x=mvco.index,
    y=mvco["MOST_rounded_momentum_flux:18.4_m:m2_s-2"],
    name="MOSTroundedMf",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)
trace0e = go.Scatter(
    x=mvco.index,
    y=np.abs(mvco["momentum_flux:18.4_m:m2_s-2"] - mvco["MOST_rounded_momentum_flux:18.4_m:m2_s-2"]),
    name="mf - MOSTroundedMf",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)
trace0f = go.Scatter(
    x=mvco.index,
    y=np.abs(mvco["momentum_flux:18.4_m:m2_s-2"] - mvco["MOST_chopped_momentum_flux:18.4_m:m2_s-2"]),
    name="mf - MOSTchoppedMf",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

print("MAE  ", np.mean( np.abs(mvco["momentum_flux:18.4_m:m2_s-2"] - mvco["MOST_momentum_flux:18.4_m:m2_s-2"])))
print("MAE rounded ", np.mean( np.abs(mvco["momentum_flux:18.4_m:m2_s-2"] - mvco["MOST_rounded_momentum_flux:18.4_m:m2_s-2"])))
print("MAE chopped ", np.mean( np.abs(mvco["momentum_flux:18.4_m:m2_s-2"] - mvco["MOST_chopped_momentum_flux:18.4_m:m2_s-2"])))


In [ ]:
trace4 = go.Scatter(
    x=mvco.index,
    y=mvco["pressure:18.4_m:hPa"],
    name="pressure",
    line=dict(color="cyan"),
    connectgaps=False,
    opacity=0.5,
)
trace5 = go.Scatter(
    x=mvco.index,
    y=mvco["sea_surface_pressure:0_m:hPa"],
    name="sea sfc P",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)



In [ ]:
trace6 = go.Scatter(
    x=mvco.index,
    y=mvco["temperature:18.4_m:K"],
    name="temp18.4m",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

trace7 = go.Scatter(
    x=mvco.index,
    y=mvco["water_sfc_temperature:0_m:K"],
    name="water surface temp",
    line=dict(color="red"),
    connectgaps=False,
    opacity=0.5,
)

trace8 = go.Scatter(
    x=mvco.index,
    y=mvco["potential_temperature:18.4_m:K"],
    name="pot temp 18.4m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace8b = go.Scatter(
    x=mvco.index,
    y=mvco["potential_temperature:0_m:K"],
    name="pot temp 0m",
    line=dict(color="orange"),
    connectgaps=False,
    opacity=0.5,
)
trace8c = go.Scatter(
    x=mvco.index,
    y=mvco["dPotTemp_dz:18.4_m:K_m-1"],
    name="dpotT/dz temp 0m",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

trace9 = go.Scatter(
    x=mvco.index,
    y=mvco["skin_virtual_potential_temperature:0_m:K"],
    name="Skin virtual potential temp",
    line=dict(color="purple"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace10 = go.Scatter(
    x=mvco.index,
    y=mvco["mixing_ratio:0_m:g_kg-1"],
    name="mixing ratio 0m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace11 = go.Scatter(
    x=mvco.index,
    y=mvco["mixing_ratio:18.4_m:g_kg-1"],
    name="mixing ratio 18.4m",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)

trace11b = go.Scatter(
    x=mvco.index,
    y=mvco["dMixingRatio_dz:18.4_m:g_kg-1_m-1"],
    name="mixing ratio 18.4m",
    line=dict(color="cyan"),
    connectgaps=False,
    opacity=0.5,
)


In [ ]:
trace12 = go.Scatter(
    x=mvco.index,
    y=mvco["relative_humidity:18.4_m:%"],
    name="rh 18.4m",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace13 = go.Scatter(
    x=mvco.index,
    y=mvco["wave_height:0_m:m"],
    name="wave height",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace14 = go.Scatter(
    x=mvco.index,
    y=mvco["wave_period:0_m:s"],
    name="wave period",
    line=dict(color="cyan"),
    connectgaps=False,
    opacity=0.5,
)
trace15 = go.Scatter(
    x=mvco.index,
    y=mvco["wave_phase_speed:0_m:m_s-1"],
    name="wave phase speed",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace16 = go.Scatter(
    x=mvco.index,
    y=mvco["wave_direction:0_m:degrees"],
    name="wave dir degrees",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace17 = go.Scatter(
    x=mvco.index,
    y=mvco["u_wave:0_m:m_s-1"],
    name="u-wave",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)
trace18 = go.Scatter(
    x=mvco.index,
    y=mvco["v_wave:0_m:m_s-1"],
    name="v-wave",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace19 = go.Scatter(
    x=mvco.index,
    y=mvco["wind_direction:18.4_m:degrees"],
    name="wind dir 18.4 m",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)


trace20 = go.Scatter(
    x=mvco.index,
    y=mvco["wind_speed:18.4_m:m_s-1"],
    name="wind_speed:18.4_m:m_s-1",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)

trace21 = go.Scatter(
    x=mvco.index,
    y=mvco['u_wind:18.4_m:m_s-1'],
    name="uwind",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)

trace22 = go.Scatter(
    x=mvco.index,
    y=mvco['v_wind:18.4_m:m_s-1'],
    name="vwind",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace19b = go.Scatter(
    x=mvco.index,
    y=mvco["sin_wind_direction:18.4_m:radians"],
    name="sin wind dir 18.4 m",
    line=dict(color="gold"),
    connectgaps=False,
    opacity=0.5,
)
trace19c = go.Scatter(
    x=mvco.index,
    y=mvco["cos_wind_direction:18.4_m:radians"],
    name="cos wind dir 18.4 m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)


In [ ]:
trace23 = go.Scatter(
    x=mvco.index,
    y=mvco["angle_between_wind_wave:0_m:degrees"],
    name="angle between wind wave",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace24 = go.Scatter(
    x=mvco.index,
    y=mvco["dT_dz:18.4_m:K_m-1"],
    name="dT/dz",
    line=dict(color="orange"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
drawTraces(*[trace24])

In [ ]:
#'dSpeed_dz:18.4_m:s-1',
trace25 = go.Scatter(
    x=mvco.index,
    y=mvco["dSpeed_dz:18.4_m:s-1"],
    name="dSpeed/dz",
    line=dict(color="orange"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
trace26 = go.Scatter(
    x=mvco.index,
    y=mvco["bulk_richardson:18.4_m:none"],
    name="bulk_richardson:18.4_m:none",
    line=dict(color="orange"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
drawTraces(*[trace26])

In [ ]:
trace27 = go.Scatter(
    x=mvco.index,
    y=mvco["u_w:18.4_m:m2_s-2"],
    name="uw cov 18.4m",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace28 = go.Scatter(
    x=mvco.index,
    y=mvco["v_w:18.4_m:m2_s-2"],
    name="vw cov 18.4m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)
trace29 = go.Scatter(
    x=mvco.index,
    y=mvco["T_w:18.4_m:C_m_s-2"], #wrong
    name="Tw cov 18.4m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace30 = go.Scatter(
    x=mvco.index,
    y=mvco["friction_velocity:18.4_m:m_s-1"],
    name="friction vel 18.4m",
    line=dict(color="red"),
    connectgaps=False,
    opacity=0.5,
)

trace31 = go.Scatter(
    x=mvco.index,
    y=mvco["temperature_scale:18.4_m:K"],
    name="temp scale 18.4m",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)

trace32 = go.Scatter(
    x=mvco.index,
    y=mvco["momentum_flux:18.4_m:m2_s-2"].where(np.fabs(mvco['momentum_flux:18.4_m:m2_s-2']) <1),
    name="momentum_flux:18.4_m:m2_s-2",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)
trace33 = go.Scatter(
    x=mvco.index,
    y=mvco["heat_flux:18.4_m:degrees_C_m_s-1"],
    name="heat_flux:18.4_m:degrees_C_m_s-1",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)


In [ ]:

drawTraces(*[trace32])

In [ ]:
#'MOST_momentum_flux:18.4_m:m2_s-1',
#'MOST_heat_flux:18.4_m:degrees_K_m_s-1'
trace34 = go.Scatter(
    x=mvco.index,
    y=mvco["MOST_momentum_flux:18.4_m:m2_s-2"],
    name="MOST mf",
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)
trace35 = go.Scatter(
    x=mvco.index,
    y=mvco["MOST_heat_flux:18.4_m:degrees_K_m_s-1"],
    name="heat_flux:18.4_m:degrees_C_m_s-1",
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

In [ ]:
drawTraces(*[trace34,trace35])

In [ ]:
# MOST related analysis 

In [ ]:
import math
bulk_richardson = "bulk_richardson:18.4_m:none"
lowestBinBound =  math.floor( mvco["bulk_richardson:18.4_m:none"].min())
highestBinBound = math.ceil(mvco["bulk_richardson:18.4_m:none"].max())
#bins = np.arange(minBulkRi, maxBulkRi, 0.01)
bins = np.arange(-2,2,.1)
# print (myNumList)
mvco.hist(column=bulk_richardson, bins=bins)

print("Unstable cases:", mvco[bulk_richardson].loc[mvco[bulk_richardson] < -0.02].count())
print("Stable cases: ", mvco[bulk_richardson].loc[mvco[bulk_richardson] > 0.02].count())
print("Neutral cases: ",mvco[bulk_richardson].loc[(mvco[bulk_richardson] >= -0.02) & (mvco[bulk_richardson] <= 0.02)].count())

In [ ]:
from matplotlib import pyplot as plt

#
# Determine the distribution of stable and unstable cases
#
unstable = mvco[bulk_richardson].loc[mvco[bulk_richardson] < -0.02].count()
stable = mvco[bulk_richardson].loc[mvco[bulk_richardson] > 0.02].count()
neutral = (mvco[bulk_richardson].loc[(mvco[bulk_richardson] >= -0.02) & (mvco[bulk_richardson] <= 0.02)].count())

data = {"unstable": unstable, "neutral": neutral, "stable": stable}
cases = list(data.keys())
values = list(data.values())

fig = plt.figure(figsize=(5, 5))

# creating the bar plot
plt.bar(cases, values, color="maroon", width=0.4)

In [ ]:
# create a test data frame with all valid inputs to MOST
mvco_lite = mvco.copy(deep=True)
print(mvco_lite.index)
mvco_lite.dropna( subset = [
        "bulk_richardson:18.4_m:none",
        "skin_virtual_potential_temperature:0_m:K",
        "wind_speed:18.4_m:m_s-1",
        "wave_height:0_m:m",
        "potential_temperature:18.4_m:K",
        "wave_phase_speed:0_m:m_s-1"], inplace = True)
print(len(mvco_lite.index))


In [ ]:
# MOST
# https://atmos.washington.edu/~breth/classes/AS547/lect/lect6.pdf
def psi_h(z, L):
    if z/L < 0:
        x = (1 - 16.0 * z / L) ** 0.25
        result = 2 * np.log(0.5 * (1 + x**2))
    else:
        result = -5.0 * z / L
    return result

In [ ]:
# MOST
# https://atmos.washington.edu/~breth/classes/AS547/lect/lect6.pdf
def psi_m(z, L):
    if z/L < 0:
        x = (1 - 16.0 * z / L) ** 0.25
        result = np.log(((1 + x**2) / 2) * ((1 + x) / 2) ** 2)- 2*np.arctan(x) + np.pi / 2
    else:
        result = -5.0 * z / L
    return result

In [ ]:
import math
import warnings
import numpy as np
from scipy.optimize import fsolve
from scipy.optimize import least_squares

#
# the next section of code solves the MOST system of equations above for friction velocity and L, Obhukov length

# Von Karman constant
k = 0.4

# The acceleration of gravity
g = 9.8

# We will count the number of systems of equations in which bulk richardson number is (-inf,-.02), [-.02, .02], (.02, inf)
# track of the convergence of systems representing unstable (Ri < -0.02), neutral, and stable ( Ri > 0.02) regimes.
riNegCount = 0
riNegConv = 0
riPosCount = 0
riPosConv = 0
riNeutCount = 0
riNeutConv = 0
totalConv = 0

# These are the solutions to our systems
mvco_lite["L"] = np.nan
mvco_lite["MOSTustar"] = np.nan
L_colIndex = mvco_lite.columns.get_loc('L')
MOSTustar_colIndex = mvco_lite.columns.get_loc('MOSTustar')

# From experience, stable regime is sensitive to the "first guess" of the root solver
# Keep track of the value of Ri in these cases. Maybe we can learn something
posNoConvRi = []
neutNoConvRi = []
negNoConvRi = []


L_array = []
for i in range( 0,200,1):
    if i == 0:
        L_array.append(.001)
    else:
        L_array.append(i)
        L_array.append(-i)

# Loop through the derived data samples, solve the system of equations as outlined above,
# record some statistics as well as the solutions
print(len(mvco_lite.index))

for i in range(0, len(mvco_lite.index)):
    if i % 1000 == 0:
        print(".", end =" ")
    # Get the known variables for this instance
    Ri = mvco_lite["bulk_richardson:18.4_m:none"].iloc[i]
    skinPotT = mvco_lite["skin_virtual_potential_temperature:0_m:K"].iloc[i]
    T = mvco_lite["water_sfc_temperature:0_m:K"].iloc[i]  # check this
    wspd = mvco_lite["wind_speed:18.4_m:m_s-1"].iloc[i]
    waveHt = mvco_lite["wave_height:0_m:m"].iloc[i]
    potT18 = mvco_lite["potential_temperature:18.4_m:K"].iloc[i]
    wavePhaseSpeed = mvco_lite["wave_phase_speed:0_m:m_s-1"].iloc[i]

    if (math.isnan(Ri)
        or math.isnan(skinPotT)
        or math.isnan(T)
        or math.isnan(wspd)
        or math.isnan(waveHt)
        or math.isnan(potT18)
        or math.isnan(wavePhaseSpeed)):
        continue

    #
    # The system of equations for which we will find the roots
    #
    def myF(z):
        ustar = z[0]
        L = z[1]

        # Surface roughness estimation
        # z0 = Hs*3.35*(u*/wavePhaseSpeed)^3.4
        #    = waveHt * 3.35 * (ustar/wavePhaseSpeed)**3.4
        #
        # F[0] = skinPotT -  thetaStar/k *(log(18.4/zt)- psi_h(z,L,Ri) - potTemp
        # F[1] = ustar/k * (log(18.4/z0) - psi_m(z,L,Ri)) - wspd
        #
        # L = -(u*)^2/(thetaStar*g/T)
        
        F = [None,None]

        # No dependence on ocean vars
        #F[0] = ustar/k * (np.log(18.4 /(.014 * ustar* ustar/g) ) - psi_m(18.4,L)) - wspd
        #F[1] = skinPotT + (ustar**2 )*skinPotT/(k*g*L) * (np.log(18.4 /(.014 * ustar*ustar/g)) - psi_h(18.4,L)) - potT18
        # dependence on ocean vars
        with np.errstate(invalid='ignore'):
            F[0] = ustar/k * (np.log(18.4 /(waveHt*3.35*(ustar/wavePhaseSpeed)**3.4) ) - psi_m(18.4,L)) - wspd
            F[1] = skinPotT + (ustar**2 )*skinPotT/(k*g*L) * (np.log(18.4 /(waveHt*3.35*(ustar/wavePhaseSpeed)**3.4)) - psi_h(18.4,L)) - potT18
        return F
        
    count = 0
    ier = -1
   
    while count < 399 and ier != 1:
        zGuess = [.1,L_array[count]]
        z, infodict, ier, mesg = fsolve(myF, zGuess, full_output=True, xtol=0.001)
        #result = least_squares(myF, zGuess, bounds=([.00001, -20000],[6,20000]), xtol=0.001)
        #if  result.status > 2:
        #    ier = 1
        #    z = result.x                                
        count = count + 1
            
            
        
    # Count the samples in each regime, the number that converge 
    if Ri < -0.02:
        riNegCount = riNegCount + 1
        if ier == 1:
            riNegConv = riNegConv + 1
        else:
            negNoConvRi.append(Ri)

    elif Ri > 0.02:
        riPosCount = riPosCount + 1
        if ier == 1:
            riPosConv = riPosConv + 1
        else:
            posNoConvRi.append(Ri)
    else:
        riNeutCount = riNeutCount + 1
        if ier == 1:
            riNeutConv = riNeutConv + 1
        else:
            neutNoConvRi.append(Ri)

    oppSignRiL = []
    if ier == 1:
        mvco_lite.iloc[i,MOSTustar_colIndex] = z[0]
        mvco_lite.iloc[i, L_colIndex] = z[1]
        if (Ri > 0 and z[1] < 0) or (Ri<0 and z[1]>0):
            oppSignRiL.append([Ri,z[1]])
        totalConv = totalConv + 1

# Output convergence statistics
print("\n")
print("unstable systems: ", riNegCount, " % convergence: ", riNegConv / riNegCount * 100)
print("neutral systems: ", riNeutCount, " % convergence: ", riNeutConv / riNeutCount * 100)
print("stable systems: ", riPosCount, " % convergence: ", riPosConv / riPosCount * 100)
print("Total: ",  len(mvco_lite.index) , " % convergence: ", totalConv/len(mvco_lite.index) *100)
print(len(oppSignRiL))


In [ ]:

traceA = go.Scatter(
    x=mvco_lite.index,
    y=mvco_lite['L'],
    name='L',
    line=dict(color="purple"),
    connectgaps=False,
    opacity=0.5,
)

traces = [ traceA]
drawTraces(*traces)

print("min: ", mvco_lite['MOSTustar'].min())
print("max: ", mvco_lite['MOSTustar'].max())

In [ ]:

histbins = np.linspace(-200, 200, 100)
mvco_lite['L'].plot.hist(bins = histbins)

print(mvco_lite['L'].min())
print(mvco_lite['L'].max())
print(mvco_lite['L'].where( abs(mvco_lite['L']) > 2000 ) .count())

In [ ]:
def mo_fluxes(ustar,L,sfcTemp):
    # specific heat at constant pressure, cp=1003.5 J kg-1K-1
    g = 9.8
    rho = 1.293  # kg/m^3 density of Pure, dry air
    # u, t = mo_similarity_offshore_branko(*args)
    mf = ustar**2
    tstar = -ustar*ustar*sfcTemp/(g*L)
    cp = 1003.5
    hf = ustar* tstar #* rho *cp

    return mf, hf


In [ ]:
mvco_lite["MOSTmomentum_flux"], mvco_lite["MOSTheat_flux"] = mo_fluxes(mvco_lite["MOSTustar"], mvco_lite["L"],mvco_lite["skin_virtual_potential_temperature:0_m:K"] )

In [ ]:
traceE = go.Scatter(
    x=mvco_lite.index,
    y=mvco_lite['MOSTmomentum_flux'],
    name='mf MOST nb',
    line=dict(color="red"),
    connectgaps=False,
    opacity=0.5,
)
traceF = go.Scatter(
    x=mvco_lite.index,
    y=mvco_lite['momentum_flux:18.4_m:m2_s-2'].where(mvco_lite['momentum_flux:18.4_m:m2_s-2']>=0),
    name='mf mx',
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

traceG = go.Scatter(
    x=mvco_lite.index,
    y=mvco_lite['MOST_momentum_flux:18.4_m:m2_s-2'],
    name='mf lib',
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)



drawTraces(*[traceE, traceF, traceG])


In [ ]:
traceH = go.Scatter(
    x=mvco_lite.index,
    y=mvco_lite['MOSTheat_flux'],
    name='hfMOST nb',
    line=dict(color="red"),
    connectgaps=False,
    opacity=0.5,
)

traceI = go.Scatter(
    x=mvco_lite.index,
    y= (mvco_lite['heat_flux:18.4_m:degrees_C_m_s-1']),
    name='hf mx',
    line=dict(color="blue"),
    connectgaps=False,
    opacity=0.5,
)

traceJ = go.Scatter(
    x=mvco_lite.index,
    y= (mvco_lite['MOST_heat_flux:18.4_m:degrees_K_m_s-1']),
    name='hf lib',
    line=dict(color="green"),
    connectgaps=False,
    opacity=0.5,
)



drawTraces(*[traceH, traceI, traceJ])
#drawTraces(*[traceI])

In [ ]:
# Hyperparameter search display
import pandas as pd
import matplotlib.pyplot as plt

# Load the data and preprocess
columns = ['Trial ID', 'Hyperparameters', 'Score', 'Status']
df = pd.read_csv('../../data/hypertuner_output/tuner_sin_cos_mf/trial_results_.csv')
df.columns = columns

# Expand hyperparameters and create separate columns
hyperparameters_dict = df['Hyperparameters'].apply(eval)
expanded_df = pd.json_normalize(hyperparameters_dict)

# Add hyperparameters columns to the dataframe
df = pd.concat([df.drop(columns=['Hyperparameters']), expanded_df], axis=1)

# Define the hyperparameters and their labels
hyperparameters = ['hidden_layers', 'hidden_neurons', 'learning_rate', 'early_stop']
color_map = plt.get_cmap('tab10')  # Get a colormap with distinct colors

# Create subplots with smaller individual plots and two columns
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(16, 12))
axes = axes.flatten()

# Increase font size globally
plt.rcParams.update({'font.size': 14})

for i, param in enumerate(hyperparameters):
    ax = axes[i]
    unique_values = df[param].unique()
    # Scatter plot for each hyperparameter
    for value in unique_values:
        subset = df[df[param] == value]
        ax.scatter(subset['Score'], subset[param], label=f'{param}: {value}', c=[color_map(i)], alpha=0.7)
    
    ax.set_xlabel('Score')
    ax.set_ylabel(param)
    ax.set_title(f'{param} vs. Score')
    ax.legend(title='Hyperparameter', loc='best')  # Legend inside the plot

# Hide any unused subplots
for i in range(len(hyperparameters), len(axes)):
    fig.delaxes(axes[i])

fig.tight_layout()
plt.show()